## import thư viện


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold

RANDOM_STATE = 42
CORRELATION_THRESHOLD = 0.98
TARGET = "Attack_type"


## Đọc dữ liệu


In [ ]:
# Đặt train.csv và validation.csv cùng thư mục với notebook.
TRAIN_PATH = r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\project\preprocessing\data\train.csv"
VALIDATION_PATH = r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\project\preprocessing\data\validation.csv"

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Target:", TARGET)

assert TARGET in train_df.columns, f"Không tìm thấy cột mục tiêu {TARGET}"
assert TARGET in validation_df.columns, f"Validation thiếu cột mục tiêu {TARGET}"
assert set(train_df.columns) == set(validation_df.columns), "Train và validation không có cùng tập cột"


## tách x và y


In [ ]:
X = train_df.drop(columns=[TARGET]).copy()
y = train_df[TARGET].copy()

X_validation = validation_df.drop(columns=[TARGET]).copy()
y_validation = validation_df[TARGET].copy()

numeric_columns = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = X.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Số biến numeric: {len(numeric_columns)}")
print(f"Số biến categorical: {len(categorical_columns)}")


## xây dựng bộ lọc tương quan


In [ ]:
class HighCorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.98):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.feature_names_in_ = X.columns.to_list()

        numeric_cols = X.select_dtypes(include=[np.number]).columns.to_list()
        self.numeric_features_ = numeric_cols

        if len(numeric_cols) < 2:
            self.drop_columns_ = []
            self.keep_columns_ = self.feature_names_in_.copy()
            self.correlated_pairs_ = pd.DataFrame(
                columns=["feature_1", "feature_2", "correlation", "abs_correlation"]
            )
            return self

        corr = X[numeric_cols].corr(method="pearson").abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

        drop_cols = []
        pairs = []

        for col in upper.columns:
            high_corr = upper.index[upper[col] > self.threshold].tolist()
            for row in high_corr:
                pairs.append({
                    "feature_1": row,
                    "feature_2": col,
                    "correlation": corr.loc[row, col],
                    "abs_correlation": corr.loc[row, col]
                })
            if high_corr:
                drop_cols.append(col)

        self.drop_columns_ = list(dict.fromkeys(drop_cols))
        self.keep_columns_ = [
            col for col in self.feature_names_in_ if col not in self.drop_columns_
        ]
        self.correlated_pairs_ = pd.DataFrame(
            pairs,
            columns=["feature_1", "feature_2", "correlation", "abs_correlation"]
        )
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        # Giữ đúng các đặc trưng đã có khi fit; không tính lại tương quan trên dữ liệu transform.
        missing = set(self.keep_columns_) - set(X.columns)
        if missing:
            raise ValueError(f"Dữ liệu transform thiếu các cột: {sorted(missing)}")
        return X.loc[:, self.keep_columns_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.keep_columns_, dtype=object)


## Kiểm tra bằng Stratified 3-Fold – fit bộ lọc riêng từng training fold


In [ ]:
skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

fold_results = []
fold_transformers = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
    X_train_fold = X.iloc[train_idx].copy()
    X_valid_fold = X.iloc[valid_idx].copy()
    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    # Quan trọng: chỉ fit trên training fold.
    corr_filter = HighCorrelationFilter(threshold=CORRELATION_THRESHOLD)
    X_train_fold_filtered = corr_filter.fit_transform(X_train_fold)

    # Validation fold chỉ được transform bằng bộ lọc đã fit ở training fold.
    X_valid_fold_filtered = corr_filter.transform(X_valid_fold)

    fold_transformers.append(corr_filter)
    fold_results.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "valid_rows": len(valid_idx),
        "numeric_before": len(corr_filter.numeric_features_),
        "removed_features": len(corr_filter.drop_columns_),
        "features_after": X_train_fold_filtered.shape[1],
        "train_shape_after": X_train_fold_filtered.shape,
        "valid_shape_after": X_valid_fold_filtered.shape
    })

    print(f"Fold {fold}")
    print("  Train sau lọc:", X_train_fold_filtered.shape)
    print("  Valid sau lọc:", X_valid_fold_filtered.shape)
    print("  Số cột bị loại:", len(corr_filter.drop_columns_))
    print("  Các cột bị loại:", corr_filter.drop_columns_)

fold_summary = pd.DataFrame(fold_results)
display(fold_summary)


## xem các cặp numeric có tương quan vượt ngưỡng theo từng fold


In [ ]:
for fold, transformer in enumerate(fold_transformers, start=1):
    print(f"Fold {fold}:")
    if transformer.correlated_pairs_.empty:
        print("  Không có cặp nào vượt ngưỡng.")
    else:
        display(
            transformer.correlated_pairs_
            .sort_values("abs_correlation", ascending=False)
            .reset_index(drop=True)
        )


## Fit bộ lọc cuối cùng trên toàn bộ train và transform validation


In [ ]:
final_corr_filter = HighCorrelationFilter(threshold=CORRELATION_THRESHOLD)

X_train_processed = final_corr_filter.fit_transform(X)
X_validation_processed = final_corr_filter.transform(X_validation)

# Giữ lại nhãn mục tiêu, không đưa nhãn vào bộ lọc tương quan.
y_train_processed = y.reset_index(drop=True)
y_validation_processed = y_validation.reset_index(drop=True)

print("Train trước lọc:", X.shape)
print("Train sau lọc:", X_train_processed.shape)
print("Validation trước lọc:", X_validation.shape)
print("Validation sau lọc:", X_validation_processed.shape)
print("Số numeric bị loại:", len(final_corr_filter.drop_columns_))
print("Danh sách cột bị loại:", final_corr_filter.drop_columns_)


## Danh sách đặc trưng sau xử lý


In [ ]:
feature_report = pd.DataFrame({
    "feature": X.columns,
    "dtype": X.dtypes.astype(str).values,
    "is_numeric": X.columns.isin(numeric_columns),
    "kept_after_filter": X.columns.isin(final_corr_filter.keep_columns_),
    "removed_by_correlation": X.columns.isin(final_corr_filter.drop_columns_)
})

display(feature_report)


In [ ]:
from pathlib import Path
import json

# Lưu đúng dữ liệu sau khi fit bộ lọc tương quan trên toàn bộ train.
processed_dir = Path(r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\exps\exp2\E08_RF\E08_Preprocessing")
processed_dir.mkdir(parents=True, exist_ok=True)

train_processed = X_train_processed.copy()
train_processed[TARGET] = y_train_processed.to_numpy()
validation_processed = X_validation_processed.copy()
validation_processed[TARGET] = y_validation_processed.to_numpy()

train_processed.to_csv(processed_dir / "train.csv.gz", index=False, compression="gzip")
validation_processed.to_csv(processed_dir / "validation.csv.gz", index=False, compression="gzip")

processing_config = {
    "target": TARGET,
    "correlation_threshold": CORRELATION_THRESHOLD,
    "feature_columns": final_corr_filter.keep_columns_,
    "removed_columns": final_corr_filter.drop_columns_,
}
(processed_dir / "E08_processing_config.json").write_text(
    json.dumps(processing_config, indent=2), encoding="utf-8"
)

print("Đã lưu train đã xử lý:", train_processed.shape)
print("Đã lưu validation đã xử lý:", validation_processed.shape)
